# ELENDRIL V6 · Grand Diagnostic Campaign — Kaggle T4×2 Blueprint

**Status: DESIGN/PREFLIGHT ONLY — NOT YET TRAINING-READY.** This notebook is a safe, structured handoff to Codex. It will **refuse expensive work** until Codex implements and commits the V6 campaign operator, pins the new source, validates the data and scorer, and the owner completes real GPU and fresh-session recovery checks. Do not assume existing V5 action notebooks implement these 15 V6 panels.

**Research goal:** identify which bottleneck limits action formation, evidence seeking, representation generalization, memory use, and eventual cumulative learning. One Kaggle experiment bundle produces 15 diagnostic panels and one ZIP. Not 15 separately launched notebooks.

**Reference Git commit:** `0cd200730d43547294cdc9bf8540a478130cce9c`. This is the PRE-V6 repository; a new immutable commit is **required** before live launch. The notebook intentionally does not accept the old SHA for campaign execution.

**Kaggle:** select GPU T4 ×2; enable Internet only to fetch pinned public source; check current quota. Each T4 has its own 16 GB memory. Intended 5–6-hour max, 35-minute result export reserve. GPU and runtime availability are not guaranteed. <https://www.kaggle.com/docs/notebooks> · <https://www.kaggle.com/product-announcements/735239>.

**Two-session safety:** a short hardware/restore qualification may be necessary before the long owner-run research campaign. A Jupyter kernel restart is not proof of fresh-VM saved-output recovery.


In [ ]:
# SINGLE OWNER CONFIGURATION CELL. All switches are disabled by default.
from pathlib import Path
REFERENCE_PRE_V6_SHA = "0cd200730d43547294cdc9bf8540a478130cce9c"
SOURCE_SHA = REFERENCE_PRE_V6_SHA # CODEX MUST REPIN THIS AFTER IT COMMITS V6 RUNNER
RUN_PREFLIGHT = False           # Only after Codex has implemented V6 and repinned SHA
RUN_CANARIES = False            # Only after source and split/correctness audit passes
RUN_RECOVERY = False            # Fresh Kaggle VM with reattached prior canary outputs
OWNER_ATTESTS_FRESH_VM = False  # Do not set true for a kernel restart
OWNER_APPROVES_CAMPAIGN = False # Explicit approval after receipts are reviewed
RUN_CAMPAIGN = False            # Expensive Kaggle experiment
RUN_EXPORT = False              # Downloadable evidence ZIP after campaign
SESSION_WALL_LIMIT_MINUTES = 360
EXPORT_RESERVE_MINUTES = 35
EXPECTED_DEVICE_COUNT = 2
OUTPUT = Path("/kaggle/working/ELENDRIL_V6")
CHECKPOINT_REATTACH_DIR = None # Set after saving a qualified prior Kaggle output as input
RUN_ID = "elendril-v6-diagnostic-001" # Fresh ID for any scientifically new campaign
assert all(isinstance(x, bool) for x in (RUN_PREFLIGHT,RUN_CANARIES,RUN_RECOVERY,OWNER_ATTESTS_FRESH_VM,OWNER_APPROVES_CAMPAIGN,RUN_CAMPAIGN,RUN_EXPORT))
print({"source":"PRE-V6 reference; repin required", "campaign_enabled":RUN_CAMPAIGN, "status":"DESIGN_ONLY"})


## 1 · Immutable scientific registry

The following registry is not a measured result. Arm, dataset and metric changes require a dated preregistration amendment before looking at any sealed test scores. The downloaded JSON copy is the complete source of truth.

In [ ]:
# This cell has no network, no disk writes and no GPU work.
import hashlib, json
from pprint import pprint
PANELS = [
  {
    "id": "P01",
    "name": "Positive-control learnability",
    "hypothesis": "Can a neural model fit explicit QUERY+EOS before OOD tests?",
    "metrics": "train loss; exact content+EOS; invalid tokens",
    "gate": "Always run; training prompts and distinct validation shown"
  },
  {
    "id": "P02",
    "name": "Known-evidence answering",
    "hypothesis": "Does it answer immediately when evidence is sufficient?",
    "metrics": "correct; redundant query count; calibration",
    "gate": "Always after P01; matched seen/unseen worlds"
  },
  {
    "id": "P03",
    "name": "Missing-evidence query",
    "hypothesis": "Does it request useful evidence when facts are missing?",
    "metrics": "query precision/recall; terminal accuracy; cost",
    "gate": "Always after P01"
  },
  {
    "id": "P04",
    "name": "Conflicting-evidence query",
    "hypothesis": "Does it avoid unsafe answers under unresolved conflicts?",
    "metrics": "query rate; wrong-answer rate; later resolution",
    "gate": "Always after P01"
  },
  {
    "id": "P05",
    "name": "Null and unavailable abstention",
    "hypothesis": "Does it distinguish explicit null vs unavailable evidence?",
    "metrics": "ABSTAIN accuracy; invalid actions; utility",
    "gate": "Always after P01"
  },
  {
    "id": "P06",
    "name": "Post-query revision",
    "hypothesis": "Can it incorporate a newly received observation and change its answer?",
    "metrics": "end-to-end episode accuracy; intervention sensitivity",
    "gate": "Always after P01"
  },
  {
    "id": "P07",
    "name": "Cost-aware thinking and queries",
    "hypothesis": "Is it better than always-query, never-query and fixed?",
    "metrics": "utility; requests; cost/quality Pareto",
    "gate": "Always after P01"
  },
  {
    "id": "P08",
    "name": "Uncertainty and self-error detection",
    "hypothesis": "Does pre-outcome confidence predict error on novel worlds?",
    "metrics": "Brier; ECE; AUROC; accuracy/coverage; extra cost",
    "gate": "Diagnostic only; fitting/calibration uses separate validation"
  },
  {
    "id": "P09",
    "name": "Legal action and EOS integrity",
    "hypothesis": "Does it emit parseable actions and terminate exactly?",
    "metrics": "exact+EOS; malformed outputs; loops; abstentions",
    "gate": "Always; raw outputs never repaired"
  },
  {
    "id": "P10",
    "name": "Query swap and query blindness",
    "hypothesis": "Does answer causally follow query, not a constant shortcut?",
    "metrics": "paired both-correct; input intervention effect",
    "gate": "Always; paired latent worlds"
  },
  {
    "id": "P11",
    "name": "Temporal/recency shortcuts",
    "hypothesis": "Does it respect timestamps and provenance over text order?",
    "metrics": "counterfactual pair accuracy; recency control",
    "gate": "Always; time/source intervention pairs"
  },
  {
    "id": "P12",
    "name": "Representation and distractor OOD",
    "hypothesis": "Does performance survive aliases, separators, new templates and distractors?",
    "metrics": "separate/joint OOD accuracy; loss; failure clusters",
    "gate": "Always; independent generator; do not train on sealed variants"
  },
  {
    "id": "P13",
    "name": "Compositional reasoning 2→3 hop",
    "hypothesis": "Does the learned core generalize beyond action tokens and 2-hop joins?",
    "metrics": "exact graph pair; 3-hop; shortcut controls",
    "gate": "Conditional on qualified route/graph task; otherwise NOT_QUALIFIED"
  },
  {
    "id": "P14",
    "name": "Frozen-weight memory benefit",
    "hypothesis": "Can admitted memory improve new episodes without a weight update?",
    "metrics": "memory on/off; retrieval choice; cost; causal swaps",
    "gate": "Conditional on available compatible learned or frozen model; provenance separate"
  },
  {
    "id": "P15",
    "name": "Sequential retention and transfer",
    "hypothesis": "Does A survive B, and does B accelerate unseen C at equal total work?",
    "metrics": "A-before/after; B; C curves; total GPU work",
    "gate": "Conditional on independently qualified A and B; otherwise NOT_QUALIFIED"
  }
]
MODEL_PROFILES = {
  "S3": {
    "width": 256,
    "layers": 4,
    "query_heads": 4,
    "kv_heads": 1,
    "head_dimension": 64,
    "ffn_width": 768,
    "context_length": 128,
    "vocabulary": "legacy_or_qualified_utf8",
    "approx_millions": 3.1
  },
  "S8": {
    "width": 384,
    "layers": 5,
    "query_heads": 6,
    "kv_heads": 2,
    "head_dimension": 64,
    "ffn_width": 1024,
    "context_length": 128,
    "vocabulary": "legacy_or_qualified_utf8",
    "approx_millions": 8.0
  },
  "S20": {
    "width": 512,
    "layers": 7,
    "query_heads": 8,
    "kv_heads": 2,
    "head_dimension": 64,
    "ffn_width": 1408,
    "context_length": 192,
    "vocabulary": "legacy_or_qualified_utf8",
    "approx_millions": 19.9
  }
}
ARMS = [
  {
    "id": "A0",
    "role": "fixed-control",
    "method": "fixed policy; always-query; never-query; random; oracle separate",
    "scale": "no-train",
    "initial_stage": "all panels where appropriate"
  },
  {
    "id": "A1",
    "role": "neural-control",
    "method": "CE AdamW, uniform exposure, established V5 core",
    "scale": "S3",
    "initial_stage": "screen"
  },
  {
    "id": "A2",
    "role": "causal-arm",
    "method": "action-weighted CE, identical examples and exposure",
    "scale": "S3",
    "initial_stage": "screen"
  },
  {
    "id": "A3",
    "role": "causal-arm",
    "method": "uncertainty-first curriculum; equal full-epoch row multiset",
    "scale": "S3",
    "initial_stage": "screen"
  },
  {
    "id": "A4",
    "role": "causal-arm",
    "method": "independent alias/template augmentation; same budget and audit",
    "scale": "S3",
    "initial_stage": "screen"
  },
  {
    "id": "A5",
    "role": "causal-arm",
    "method": "learned decision or evidence-seeking regularizer; explicit objective and baseline",
    "scale": "S3",
    "initial_stage": "screen-if-qualified"
  },
  {
    "id": "A6",
    "role": "conditional-scale",
    "method": "winning S3 mechanism; scale probe",
    "scale": "S8",
    "initial_stage": "only-after-screen"
  },
  {
    "id": "A7",
    "role": "conditional-scale",
    "method": "winning S3 mechanism; scale probe",
    "scale": "S20",
    "initial_stage": "only-after-efficiency-gate"
  },
  {
    "id": "A8",
    "role": "conditional-confirmation",
    "method": "best two preregistered S3 competitors, new seeds and sealed generator",
    "scale": "S3",
    "initial_stage": "only-after-screen"
  }
]
EXPECTED_REGISTRY_SHA256 = "dcbc1be6acb28a42a89fe63078d066339c56682fa66e67a629e2e4c9c3562abe"
print("15 planned panels:")
for row in PANELS: print(f"{row['id']} · {row['name']}")
print("Expected scales:",[(name, config['approx_millions']) for name,config in MODEL_PROFILES.items()])


## 2 · Pinned checkout and real device admission (owner-enabled)

This is intentionally fail-closed. Only a committed, repinned V6 operator can pass. The notebook never `pip install`s a replacement CUDA stack. Native PyTorch and two physical T4 UUIDs must be inspected, with per-device free VRAM and disk checked before canary.

In [ ]:
# NOTE: will not perform network access or create files with defaults.
import os, re, subprocess, sys, shutil
from pathlib import Path
REPOSITORY = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
REPO = Path('/kaggle/working/elendril-v6-pinned-source')
ENVIRONMENT = None
if RUN_PREFLIGHT:
    if SOURCE_SHA == REFERENCE_PRE_V6_SHA or not re.fullmatch(r"[0-9a-f]{40}", SOURCE_SHA):
        raise RuntimeError("NO-GO: Codex must implement and pin the NEW immutable V6 source commit")
    if RUN_CAMPAIGN and not OWNER_APPROVES_CAMPAIGN:
        raise RuntimeError("NO-GO: explicit campaign approval missing")
    os.environ['PYTHONDONTWRITEBYTECODE']='1'
    os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
    if not REPO.exists():
        REPO.mkdir(parents=True)
        for command in (["git","init","-q"], ["git","remote","add","origin",REPOSITORY], ["git","fetch","--depth=1","origin",SOURCE_SHA], ["git","checkout","--detach",SOURCE_SHA]):
            subprocess.run(command, cwd=REPO, check=True, timeout=240)
    head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
    if head != SOURCE_SHA or subprocess.check_output(['git','status','--porcelain','--untracked-files=all'],cwd=REPO,text=True).strip():
        raise RuntimeError('NO-GO: source checkout is not exact and clean')
    sys.path.insert(0, str(REPO))
    from tools import elendril_v6_campaign as v6
    if Path(v6.__file__).resolve() != (REPO/'tools'/'elendril_v6_campaign.py').resolve():
        raise RuntimeError('NO-GO: operator not loaded from pinned source')
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count()!=2:
        raise RuntimeError('NO-GO: require two visible CUDA devices')
    raw=subprocess.check_output(['nvidia-smi','--query-gpu=index,name,uuid,memory.free,memory.total,driver_version','--format=csv,noheader,nounits'],text=True)
    info=[dict(zip(['index','name','uuid','free_mib','total_mib','driver'],map(str.strip,line.split(',')))) for line in raw.splitlines() if line.strip()]
    if len(info)!=2 or len({g['uuid'] for g in info})!=2 or any('T4' not in g['name'].upper() or int(g['free_mib'])<11000 for g in info):
        raise RuntimeError('NO-GO: need 2 distinct physical T4s each with >= 11 GiB free')
    if not OUTPUT.parent.exists() or shutil.disk_usage(OUTPUT.parent).free < 4*(1<<30):
        raise RuntimeError('NO-GO: Kaggle working disk insufficient')
    ENVIRONMENT=v6.preflight(source=SOURCE_SHA,protocol_sha=EXPECTED_REGISTRY_SHA256,devices=info,run_id=RUN_ID)
    if ENVIRONMENT['status']!='PASS': raise RuntimeError('NO-GO: V6 preflight failed')
    print(ENVIRONMENT)
else:
    print('PREFLIGHT_DISABLED: no repo download, no GPU access')


## 3 · Cloud canary and durable checkpoint challenge

Codex must implement these explicit entrypoints with tests. The canary measures real GPU timing, exact EOS, model/optimizer counts, checkpoint hash/readback and restart. A separate new VM reattaches a saved output and proves recovery; simply rerunning a cell in the same VM does not count.

In [ ]:
CANARY_RECEIPTS=None
RECOVERY_RECEIPTS=None
if RUN_CANARIES:
    if not RUN_PREFLIGHT or ENVIRONMENT is None: raise RuntimeError('NO-GO: preflight first')
    CANARY_RECEIPTS=v6.run_canaries(source=SOURCE_SHA,protocol_sha=EXPECTED_REGISTRY_SHA256,
        preflight=ENVIRONMENT,output=OUTPUT/'qualification',per_device=True)
    if CANARY_RECEIPTS.get('status')!='PASS':raise RuntimeError('NO-GO: hardware canary did not pass')
    print('SAVE this entire qualification output; end VM, reattach as Kaggle input, open a NEW VM')
else:
    print('CANARIES_DISABLED')
if RUN_RECOVERY:
    if not RUN_PREFLIGHT or not OWNER_ATTESTS_FRESH_VM or CHECKPOINT_REATTACH_DIR is None:
        raise RuntimeError('NO-GO: true fresh-VM reattachment evidence is required')
    RECOVERY_RECEIPTS=v6.verify_off_session_recovery(source=SOURCE_SHA,
        prior_dir=Path(CHECKPOINT_REATTACH_DIR),current_environment=ENVIRONMENT,
        output=OUTPUT/'recovery',must_resume_optimizer=True)
    if RECOVERY_RECEIPTS.get('status')!='PASS':raise RuntimeError('NO-GO: off-session recovery failed')
else:
    print('RECOVERY_DISABLED')


## 4 · Owner-run coordinated training

All expensive toggles remain disabled. The V6 operator must implement a stage budget allocator, two independent T4 worker lanes, preregistered paired arms, optional S8/S20 only after positive S3 signal, 15 panels, failed-arm preservation and an export-time reserve. No network/service billing or local training is authorized.

In [ ]:
CAMPAIGN_RESULT=None
if RUN_CAMPAIGN:
    if not (RUN_PREFLIGHT and RUN_CANARIES and RUN_RECOVERY and OWNER_APPROVES_CAMPAIGN):
        raise RuntimeError('NO-GO: owner approval, canaries, preflight and off-session recovery required')
    if CANARY_RECEIPTS is None or RECOVERY_RECEIPTS is None:
        raise RuntimeError('NO-GO: no qualification receipts')
    if SESSION_WALL_LIMIT_MINUTES < EXPORT_RESERVE_MINUTES + 60 or SESSION_WALL_LIMIT_MINUTES>600:
        raise ValueError('NO-GO: unsafe wall limit')
    CAMPAIGN_RESULT = v6.run_campaign(source=SOURCE_SHA,
        protocol_sha=EXPECTED_REGISTRY_SHA256,preflight=ENVIRONMENT,
        canary=CANARY_RECEIPTS,recovery=RECOVERY_RECEIPTS,
        run_id=RUN_ID,output=OUTPUT,
        wall_limit_seconds=SESSION_WALL_LIMIT_MINUTES*60,
        export_reserve_seconds=EXPORT_RESERVE_MINUTES*60,
        preserve_failed_and_partial=True)
    if CAMPAIGN_RESULT.get('status') not in {'COMPLETE','PARTIAL','FAILED_WITH_RECEIPTS'}:
        raise RuntimeError('NO-GO: campaign returned unrecognized completion status')
    print(CAMPAIGN_RESULT)
else:
    print('CAMPAIGN_DISABLED — registry/preflight scaffold only')


## 5 · Complete evidence ZIP

Export is separately owner-controlled. This cell may bundle partial or failed results but cannot relabel them as complete. Codex must implement independent rescore and SHA manifests before enabling this path. Kaggle output must be saved as a version so the ZIP survives the session.

In [ ]:
ZIP_PATH=None
if RUN_EXPORT:
    if not RUN_PREFLIGHT or CAMPAIGN_RESULT is None:
        raise RuntimeError('NO-GO: no actual campaign results')
    report = v6.independent_rescore_and_export(
        source=SOURCE_SHA, protocol_sha=EXPECTED_REGISTRY_SHA256,
        campaign=CAMPAIGN_RESULT, output=OUTPUT,
        zip_name='ELENDRIL_V6_RESULTS.zip',include_all_failures=True)
    ZIP_PATH=Path(report['zip_path'])
    if not ZIP_PATH.is_file() or report.get('status')!='EXPORT_VERIFIED':
        raise RuntimeError('NO-GO: ZIP export failed full-manifest verification')
    print('SAVED OUTPUT ZIP:',ZIP_PATH)
else:
    print('EXPORT_DISABLED — no fabricated ZIP or results')


## Scientific interpretation rules

A failed positive-control overfit test indicts objective, tokenizer, gradient or checkpoint mechanics before any scale experiment. Correct familiar actions but poor aliases suggest representation/data overfit. Good accuracy but wasteful queries requires cost-aware policy work. Two-hop success but three-hop failure indicates depth limits. Fixed-retrieval success does not imply a learned retrieval policy. Sequential retention without acquired B is not cumulative learning. Always preserve negatives and full raw receipts.

**This draft notebook is not yet authorized for Kaggle training. Codex must implement V6 runner, 15 panels, tests, regenerate the notebook against a new source SHA, and present a hardware qualification report.**